In [1]:
!pip install sagemaker xgboost pandas numpy matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 90.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# This is to load dataset
data = pd.read_csv('Business analytics ready dataset (Final Project).csv')

# This is to select features and target
features = ['qty', 'freight_price', 'comp_1', 'ps1', 'fp1', 'comp_2', 'ps2', 'fp2', 'comp_3', 'ps3', 'fp3', 'lag_price']
target = 'unit_price'

X = data[features]
y = data[target]

# This is to handle missing values
X = X.fillna(X.mean())

# This is to train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# This is to standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# This is to Train XGBoost Model
xgb_model = XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train_scaled, y_train)

Matplotlib is building the font cache; this may take a moment.
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [3]:
import joblib

# This is to save the model
model_filename = 'xgboost_model.pkl'
joblib.dump(xgb_model, model_filename)

['xgboost_model.pkl']

In [4]:
import boto3

# This is to initialize S3 client
s3 = boto3.client('s3')

# This is to upload the model to your S3 bucket
bucket_name = 'amazon-sagemaker-162288433177-us-east-1-1636cf756751'  
s3.upload_file(model_filename, bucket_name, 'xgboost_model.pkl')

In [12]:
import joblib
import tarfile

# This is to compress the model into a tar.gz file (SageMaker requirement)
with tarfile.open('xgboost_model.tar.gz', mode='w:gz') as archive:
    archive.add('xgboost_model.pkl')


In [15]:
import tarfile

# This to add inference.py and the model to a new .tar.gz file
with tarfile.open('xgboost_model.tar.gz', mode='w:gz') as archive:
    archive.add('xgboost_model.pkl')
    archive.add('inference.py')

In [17]:
import boto3

s3 = boto3.client('s3')

s3.upload_file('xgboost_model.tar.gz', 
               'amazon-sagemaker-162288433177-us-east-1-1636cf756751', 
               'xgboost_model.tar.gz')

In [ ]:
import sagemaker
from sagemaker.xgboost import XGBoostModel

# This is the sageMaker Session
sagemaker_session = sagemaker.Session()

# This is the XGBoost Model
xgb_model = XGBoostModel(
    model_data='s3://amazon-sagemaker-162288433177-us-east-1-1636cf756751/xgboost_model.tar.gz',
    role='arn:aws:iam::162288433177:role/SageMakerExecutionRole',
    entry_point='inference.py',  
    framework_version='1.5-1',
    sagemaker_session=sagemaker_session
)

# This is to deploy the model to an endpoint
predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large'
)

[03/17/25 11:11:36] INFO     Ignoring unnecessary instance type: ml.m5.large.                     ]8;id=28208;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=971817;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#530\530]8;;\

                    INFO     Creating model with name: sagemaker-xgboost-2025-03-17-11-11-36-831    ]8;id=6517;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=90433;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#4094\4094]8;;\

[03/17/25 11:11:37] INFO     Creating endpoint-config with name                                     ]8;id=165503;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=931043;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#5889\5889]8;;\
                             sagemaker-xgboost-2025-03-17-11-11-37-379                                             

                    INFO     Creating endpoint with name sagemaker-xgboost-2025-03-17-11-11-37-379  ]8;id=58269;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=25675;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#4711\4711]8;;\

----